# PDF → Word para Kindle

Converte um PDF em um `.docx` **refluível e diagramado para ler no Kindle**:
parágrafos remontados, títulos, sumário navegável e notas reunidas no fim.

Funciona pelo navegador do celular — nada é instalado no aparelho, e o arquivo
não passa por nenhum servidor meu: tudo acontece na máquina temporária que o
Google empresta para este notebook.

**Toque no ▶ de cada célula, de cima para baixo.**


In [ ]:
#@title Passo 1 — Instalar (demora ~1 minuto) { display-mode: "form" }
!pip install -q "git+https://github.com/brunobracco-hash/Mobile.git"
print("Pronto. Pode ir para o passo 2.")


In [ ]:
#@title Passo 2 (opcional) — Só se o livro for escaneado { display-mode: "form" }
# Um PDF escaneado é a fotografia das páginas: não tem texto, só imagem.
# Esta célula instala o reconhecimento de texto (demora ~2 minutos).
# Se o seu PDF é um e-book comum, pule para o passo 3.
!apt-get -qq install -y ocrmypdf tesseract-ocr-por > /dev/null 2>&1
!ocrmypdf --version && echo "OCR pronto."


In [ ]:
#@title Passo 3 — Enviar o PDF { display-mode: "form" }
from google.colab import files

enviados = files.upload()
for nome in enviados:
    print("recebido:", nome)


In [ ]:
#@title Passo 4 — Converter e baixar { display-mode: "form" }
titulo = ""  #@param {type:"string"}
autor = ""  #@param {type:"string"}
notas_de_rodape = "reunir no fim"  #@param ["reunir no fim", "manter onde estão", "descartar"]
livro_escaneado = "detectar sozinho"  #@param ["detectar sozinho", "sempre reconhecer o texto", "nunca reconhecer"]
manter_imagens = True  #@param {type:"boolean"}
capitulo_em_pagina_nova = True  #@param {type:"boolean"}
texto_justificado = True  #@param {type:"boolean"}

import glob
import os

from pdf2kindle import Options, convert

NOTAS = {"reunir no fim": "end", "manter onde estão": "inline", "descartar": "drop"}
OCR = {"detectar sozinho": "auto", "sempre reconhecer o texto": "force", "nunca reconhecer": "off"}

opcoes = Options(
    title=titulo or None,
    author=autor or None,
    footnotes=NOTAS[notas_de_rodape],
    ocr=OCR[livro_escaneado],
    keep_images=manter_imagens,
    page_break_chapters=capitulo_em_pagina_nova,
    justify=texto_justificado,
)

pdfs = sorted(glob.glob("*.pdf"))
if not pdfs:
    print("Nenhum PDF por aqui. Volte ao passo 3 e envie o arquivo.")

for pdf in pdfs:
    saida = os.path.splitext(pdf)[0] + ".docx"
    resultado = convert(pdf, saida, opcoes)
    s = resultado.document.stats
    print(f"\n{pdf} → {saida}")
    print(
        f"  {s['pages']} páginas · {s['headings']} títulos · {s['paragraphs']} parágrafos"
        f" · {s['images']} imagens · {s['notes']} notas"
    )
    if resultado.ocr_applied:
        print("  texto reconhecido por OCR")
    for aviso in resultado.warnings:
        print(f"  aviso: {aviso}")

    try:
        from google.colab import files

        files.download(saida)
    except Exception:
        print("  (se o download não começar, abra a pasta 📁 na lateral e baixe por lá)")


## Como mandar para o Kindle

1. O `.docx` foi baixado para o seu celular (normalmente em *Downloads*).
2. Anexe em um e-mail para o **seu endereço `@kindle.com`** — ele aparece em
   *Amazon → Conta → Conteúdo e dispositivos → Preferências → Enviar para Kindle*.
   Mande do e-mail que estiver autorizado nessa mesma página.
3. Em poucos minutos o livro aparece no aplicativo Kindle, já com o sumário
   navegável montado a partir dos títulos.

Dá também para abrir o `.docx` no Word ou no Google Docs antes de enviar, se
quiser conferir ou corrigir algum título — eles usam os estilos `Título 1/2/3`.

**Livro escaneado:** as figuras se perdem, porque fazem parte da fotografia da
página. O texto vem completo.
